In [1]:
import glob
import os
import pickle
import numpy as np

In [2]:
with open('episodic_data/qubit_control/qubit_control_memoryless_seed42.pkl', 'rb') as f:
    data = pickle.load(f)

print("Type:", type(data))
print("Length:", len(data))

# Inspect the first entry to find fidelity-related keys
first = data[0]
print("First entry keys:", list(first.keys()))
for k, v in first.items():
    if 'fid' in k.lower():
        import numpy as np
        arr = np.array([d[k] for d in data])
        print(f"Key '{k}': shape = {arr.shape}, dtype = {arr.dtype}")

Type: <class 'list'>
Length: 5000
First entry keys: ['timestep', 'mean_reward', 'mean_fidelity', 'max_fidelity', 'min_fidelity', 'std_fidelity', 'fraction_solved', 'mean_omega_x', 'std_omega_x']
Key 'mean_fidelity': shape = (5000,), dtype = float32
Key 'max_fidelity': shape = (5000,), dtype = float32
Key 'min_fidelity': shape = (5000,), dtype = float32
Key 'std_fidelity': shape = (5000,), dtype = float32


In [3]:
BASE = "episodic_data/qubit_control"


def relative_dip_metric(mean_F, window=100):
    """
    For each update t, compute (peak so far - current F).
    The 'worst dip' is the maximum drop from any prior peak.
    """
    running_max = np.maximum.accumulate(mean_F)
    drops = running_max - mean_F
    return {
        'worst_drop': float(drops.max()),
        'worst_drop_at': int(drops.argmax()),
        'peak_before_drop': float(running_max[drops.argmax()]),
        'low_at_drop': float(mean_F[drops.argmax()]),
    }

In [5]:
seeds = {"seed42", "seed43", "seed44"}

for path in sorted(glob.glob(os.path.join(BASE, "*.pkl"))):
    if not any(s in os.path.basename(path) for s in seeds):
        continue

    with open(path, "rb") as f:
        data = pickle.load(f)

    mean_F = np.array([d['mean_fidelity'] for d in data], dtype=float)
    m = relative_dip_metric(mean_F)
    name = os.path.basename(path)
    print(
        f"for {name}, "
        f"worst_drop={m['worst_drop']:.4f}, "
        f"worst_drop_at={m['worst_drop_at']}, "
        f"peak_before_drop={m['peak_before_drop']:.4f}, "
        f"low_at_drop={m['low_at_drop']:.4f}"
    )

for qubit_control_context50_seed42.pkl, worst_drop=0.3646, worst_drop_at=4722, peak_before_drop=0.9931, low_at_drop=0.6286
for qubit_control_context50_seed43.pkl, worst_drop=0.4418, worst_drop_at=641, peak_before_drop=0.9892, low_at_drop=0.5474
for qubit_control_context50_seed44.pkl, worst_drop=0.3525, worst_drop_at=4741, peak_before_drop=0.9934, low_at_drop=0.6409
for qubit_control_context5_seed42.pkl, worst_drop=0.5432, worst_drop_at=583, peak_before_drop=0.9963, low_at_drop=0.4531
for qubit_control_context5_seed43.pkl, worst_drop=0.5746, worst_drop_at=121, peak_before_drop=0.9927, low_at_drop=0.4181
for qubit_control_context5_seed44.pkl, worst_drop=0.8189, worst_drop_at=163, peak_before_drop=0.9939, low_at_drop=0.1750
for qubit_control_memoryless_seed42.pkl, worst_drop=0.5316, worst_drop_at=373, peak_before_drop=0.9959, low_at_drop=0.4643
for qubit_control_memoryless_seed43.pkl, worst_drop=0.5696, worst_drop_at=545, peak_before_drop=0.9922, low_at_drop=0.4226
for qubit_control_memor